# BeautyCore scheduled demo starter

Use the tested pinned Kaggle image, T4 GPU and Internet. Attach both private binary inputs and enable the three required Secrets. Run the three code cells in order, once per session. This notebook embeds the corrected CPU startup; the old private archives and model checks stay unchanged. See the client handover guide. Fresh Kaggle end-to-end rehearsal of this consolidated starter is still required.


In [ ]:
"""Verified private bundles and early host preflight for the client demo notebook."""
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
import json, platform, socket, sys, uuid

BUNDLES = (
    ("capstone_train002_20260928_v1.bin", "f0d1fed52e00b6c5a1da2cade3731237675cdebbfc0b2cde3b2dbda99a4019ac", "gpu-src", "gate3_bundle.json"),
    ("render_kaggle_cpu_20261009.bin", "f391050851519e9cc1080830b615a6cc457b6dd1e21fde156522bb959fad9ede", "cpu-src", "demo_bundle.json"),
)

def extract(source, expected, target, manifest_name):
    if sha256(source.read_bytes()).hexdigest() != expected:
        raise RuntimeError("Bundle hash differs: " + source.name)
    if target.exists():
        raise RuntimeError("Extraction requires a new directory; existing source is never overwritten")
    with ZipFile(source) as archive:
        manifest = json.loads(archive.read(manifest_name))
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(manifest["files"]) | {manifest_name}:
            raise RuntimeError("Bundle member list differs")
        if archive.testzip() is not None:
            raise RuntimeError("Bundle archive integrity failed")
        # Validate every member before writing any of the archive contents.
        members = []
        for name in names:
            destination = (target / name).resolve()
            if ":" in name or chr(92) in name or not destination.is_relative_to(target.resolve()):
                raise RuntimeError("Unsafe bundle member path")
            raw = archive.read(name)
            if name != manifest_name and sha256(raw).hexdigest() != manifest["files"][name]:
                raise RuntimeError("Bundle member hash differs: " + name)
            members.append((destination, raw))
        for destination, raw in members:
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(raw)

def check_host(observed, expected):
    if any(observed.get(key) != expected.get(key) for key in ("python", "torch", "cuda", "gpu")):
        raise RuntimeError("Kaggle image differs from the reviewed GPU image. Expected "
                           + json.dumps(expected) + "; observed " + json.dumps(observed)
                           + ". Use the tested pinned notebook; do not change the model guards.")

def prepare(inputs=Path("/kaggle/input"), working=Path("/kaggle/working")):
    if sys.platform != "linux":
        raise RuntimeError("Run this starter inside Kaggle, not on the client device")
    for port in (8765, 8800):
        with socket.socket() as connection:
            connection.settimeout(1)
            if connection.connect_ex(("127.0.0.1", port)) == 0:
                raise RuntimeError("An AI service is already running. Do not run setup again.")
    work = working / ("render-kaggle-client-" + uuid.uuid4().hex)
    for name, digest, directory, manifest in BUNDLES:
        matches = list(inputs.rglob(name))
        if len(matches) != 1:
            raise RuntimeError("Attach exactly one private input containing " + name)
        extract(matches[0], digest, work / directory, manifest)
    import torch
    observed = {"python": platform.python_version(), "torch": torch.__version__,
                "cuda": torch.version.cuda,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    review = json.loads((work / "gpu-src/docs/experiments/unified-kaggle-gate2-review.json").read_text())
    expected = {key: review["environment"][key] for key in observed}
    check_host(observed, expected)
    print("Private bundles and reviewed GPU host verified.", flush=True)
    print("Working directory:", work, flush=True)
    print("Run Cell 2 once. Keep this Kaggle session running during the demo.", flush=True)
    return work

CLIENT_WORK = prepare()


In [ ]:
# Stable website configuration; never put keys in cells.
import os
BEAUTYCORE_PUBLIC_URL = 'https://beautycore-demo.onrender.com'
os.environ['BEAUTYCORE_PUBLIC_URL'] = BEAUTYCORE_PUBLIC_URL
(CLIENT_WORK / 'runtime_registration.py').write_text('"""HMAC metadata registration/heartbeat. No credentials or generation requests in logs."""\nfrom hashlib import sha256\nimport hmac\nimport json\nimport os\nfrom pathlib import Path\nimport socket\nimport sys\nimport time\nfrom urllib.error import HTTPError, URLError\nfrom urllib.parse import urlsplit\nfrom urllib.request import Request, HTTPRedirectHandler, build_opener\n\nSCHEMA = "beautycore-runtime-v1"\nHEARTBEAT_SECONDS = 60\n\nclass RegistrationError(RuntimeError):\n    def __init__(self, status=None):\n        self.status = status\n        super().__init__("Website connection not acknowledged" + (f" (HTTP {status})" if status else ""))\n\nclass NoRedirect(HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        return None\n\ndef canonical(message):\n    return json.dumps(message, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode()\n\ndef sign(message, key):\n    derived = hmac.new(key.encode(), b"beautycore-runtime-key-v1", sha256).digest()\n    return hmac.new(derived, canonical(message), sha256).hexdigest()\n\ndef website_root(website):\n    p = urlsplit(website)\n    if (p.scheme != "https" or not p.hostname or p.username or p.password or p.port\n            or p.query or p.fragment or p.path.rstrip("/")):\n        raise RegistrationError()\n    return website.rstrip("/")\n\ndef send(website, record, key, operation, *, opener=None, now=time.time):\n    if len(key) < 32:\n        raise RegistrationError()\n    message = {"schema": SCHEMA, "operation": operation, "session_id": record["session_id"],\n               "endpoint": record["url"], "started_at": record["started_at"], "issued_at": int(now())}\n    request = Request(website_root(website) + "/api/internal/ai-runtime/register", data=canonical(message),\n                      headers={"Content-Type": "application/json", "x-ai-runtime-signature": sign(message, key)})\n    try:\n        with (opener or build_opener(NoRedirect())).open(request, timeout=75) as response:\n            if response.status != 200:\n                raise RegistrationError(response.status)\n            raw = response.read(8193)\n            if len(raw) > 8192:\n                raise RegistrationError()\n            ack = json.loads(raw)\n            if (ack.get("connected") is not True or ack.get("session_id") != record["session_id"]\n                    or not isinstance(ack.get("expires_at"), (int, float)) or ack["expires_at"] <= now() * 1000):\n                raise RegistrationError()\n            return ack\n    except HTTPError as error:\n        raise RegistrationError(error.code) from None\n    except (URLError, OSError, TimeoutError, ValueError, TypeError, AttributeError):\n        raise RegistrationError() from None\n\ndef register(website, record, key, *, send_fn=send, sleep=time.sleep, attempts=4):\n    # Metadata only. The GPU request is never repeated here. Accommodates Render cold start.\n    for attempt in range(attempts):\n        try:\n            return send_fn(website, record, key, "register")\n        except RegistrationError as error:\n            if error.status in (400, 401, 403, 409, 404) or attempt == attempts - 1:\n                raise\n            print("Website registration waiting; AI processes remain running.", flush=True)\n            sleep(10)\n\ndef services_alive(record):\n    try:\n        for field in ("api_pid", "tunnel_pid"):\n            os.kill(record[field], 0)\n        for port in (8765, 8800):\n            with socket.socket() as connection:\n                connection.settimeout(2)\n                if connection.connect_ex(("127.0.0.1", port)) != 0:\n                    return False\n        return True\n    except (OSError, KeyError, TypeError):\n        return False\n\ndef heartbeat(website, record, key, *, send_fn=send, alive=services_alive, sleep=time.sleep, now=time.time):\n    acknowledged_until = record["registration_expires_at"]\n    while True:\n        sleep(HEARTBEAT_SECONDS)\n        if not alive(record):\n            print("AI process stopped. Website lease will expire; start a fresh notebook session.", flush=True)\n            return\n        try:\n            ack = send_fn(website, record, key, "renew")\n            acknowledged_until = ack["expires_at"]\n        except RegistrationError as error:\n            if error.status in (400, 401, 403, 409, 404) or now() * 1000 >= acknowledged_until:\n                print("Website lease expired or refused. Start a fresh session; do not replay generation.", flush=True)\n                return\n            print("Website heartbeat not acknowledged; existing lease has not been extended.", flush=True)\n\nif __name__ == "__main__":\n    record = json.loads(Path(sys.argv[2]).read_text())\n    heartbeat(sys.argv[1], record, os.environ.get("AI_BACKEND_API_KEY", ""))\n', encoding='utf-8')
"""New scheduled-demo runner. Existing notebooks and GPU bundle are not modified."""
import argparse
import json
import os
from pathlib import Path
import re
import socket
import subprocess
import sys
import time
from uuid import uuid4
from urllib.request import Request, urlopen

WORK = CLIENT_WORK
GPU_ROOT = WORK / "gpu-src"
CPU_ROOT = WORK / "cpu-src"
PORT = 8800


def command(args, cwd, env, stage):
    print("DEMO STAGE:", stage, flush=True)
    log = WORK / (stage + ".log")
    with log.open("w") as output:
        process = subprocess.Popen(args, cwd=cwd, env=env, stdout=output, stderr=subprocess.STDOUT)
        while process.poll() is None:
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                print(stage, "still running; log bytes:", log.stat().st_size, flush=True)
    if process.returncode:
        raise RuntimeError(stage + " failed; inspect its log")


def read_json(url, key=None):
    headers = {"x-ai-backend-key": key, "x-ai-user-id": "demo-operator"} if key else {}
    with urlopen(Request(url, headers=headers), timeout=40) as response:
        return json.load(response)


def main():
    if sys.platform != "linux":
        raise RuntimeError("Run in the new Kaggle Linux GPU notebook")
    if (WORK / "demo-session.json").exists():
        raise RuntimeError("Refuse duplicate startup; use a fresh notebook session")
    for port in (8765, PORT):
        with socket.socket() as connection:
            connection.settimeout(1)
            if connection.connect_ex(("127.0.0.1", port)) == 0:
                raise RuntimeError("An AI service is already running; do not start another worker")
    # Total RAM, not VRAM. Old GPU evidence peaked near 23 GiB before CPU vision.
    memory = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text())[1]) * 1024
    if memory < 30 * 1024**3:
        raise RuntimeError("At least 30 GiB host RAM required for this unverified combined workload")
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    backend_key = secrets.get_secret("AI_BACKEND_API_KEY") or ""
    gpu_key = secrets.get_secret("AI_REMOTE_API_KEY") or ""
    gemini_key = secrets.get_secret("GEMINI_API_KEY") or ""
    if len(backend_key) < 32 or len(gpu_key) < 24 or not gemini_key:
        raise RuntimeError("Enable the three required Kaggle Secrets; never put keys into cells")
    session_id = str(uuid4())
    started_at = int(time.time())
    # Reserve the session before starting any subprocess; failed sessions need a fresh restart.
    (WORK / "demo-session.json").write_text(json.dumps({"status": "starting", "host_ram_bytes": memory}))
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    # CPU services do not use Kaggle's inline-notebook plotting backend.
    cpu_env = {**env, "CUDA_VISIBLE_DEVICES": "", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "MPLBACKEND": "Agg"}
    import ctypes.util
    if not ctypes.util.find_library("portaudio"):
        command(["apt-get", "update"], CPU_ROOT, env, "portaudio-index")
        command(["apt-get", "install", "-y", "libportaudio2"], CPU_ROOT, env, "portaudio-install")
    for name in ("backend-env", "segment-env"):
        # Kaggle ensurepip can fail even on the reviewed Python image.
        # Use host pip only as an installer targeting the isolated CPU environment.
        command([sys.executable, "-m", "venv", "--without-pip", str(WORK / name)], CPU_ROOT, cpu_env, name + "-create")
        command([sys.executable, "-m", "pip", "--python", str(WORK / name), "install", "pip==25.3"],
                CPU_ROOT, cpu_env, name + "-pip-bootstrap")
    api_python = WORK / "backend-env/bin/python"
    seg_python = WORK / "segment-env/bin/python"
    command([str(api_python), "-m", "pip", "install", "-r", "backend/requirements-nails.txt", "--constraint", "backend/requirements-demo-constraints.txt"],
            CPU_ROOT, cpu_env, "backend-install")
    command([str(seg_python), "-m", "pip", "install", "torch==2.10.0", "torchvision==0.25.0", "--index-url", "https://download.pytorch.org/whl/cpu"],
            CPU_ROOT, cpu_env, "segment-torch-install")
    command([str(seg_python), "-m", "pip", "install", "-r", "backend/requirements-nails-segmenter.txt"],
            CPU_ROOT, cpu_env, "segment-install")
    for name, interpreter in (("backend", api_python), ("segment", seg_python)):
        command([str(interpreter), "-m", "pip", "check"], CPU_ROOT, cpu_env, name + "-pip-check")
    checkpoint = CPU_ROOT / "data/nails/checkpoints/mnemic/nails_seg_s_yolov8_v1.pt"
    command([str(seg_python), "-c", "import torch, torchvision; from ultralytics import YOLO; assert torchvision.extension._has_ops(); assert '+cpu' in torch.__version__; YOLO(" + repr(str(checkpoint)) + ")"],
            CPU_ROOT, cpu_env, "segment-load")
    landmark = CPU_ROOT / "data/nails/checkpoints/mediapipe/hand_landmarker.task"
    cpu_env["PYTHONPATH"] = str(CPU_ROOT / "backend")
    command([str(api_python), "-c", "from pathlib import Path; from app.nails.geometry import MediaPipeHandLocalizer; MediaPipeHandLocalizer(Path(" + repr(str(landmark)) + "))"],
            CPU_ROOT, cpu_env, "landmark-load")
    gpu_code = f"""from pathlib import Path
from scripts import unified_gate3_bootstrap as original
from scripts.capstone_train002_bootstrap import worker_environment
start = original.start_server
original.start_server = lambda output, env: start(output, worker_environment(env))
report = original.bootstrap(Path({str(WORK / 'gpu-session')!r}), local_only=True)
assert report['status'] == 'READY_FOR_LIVE_ACCEPTANCE', 'GPU bootstrap failed'
"""
    # Preserves exact host/package/model/adapter checks; one GPU load, no public GPU tunnel.
    command([sys.executable, "-c", gpu_code], GPU_ROOT,
            {**env, "AI_REMOTE_API_KEY": gpu_key}, "gpu-bootstrap")
    cpu_env.update(AI_BACKEND_API_KEY=backend_key, AI_GPU_LOOPBACK="1",
                   AI_REMOTE_URL="http://127.0.0.1:8765", AI_REMOTE_API_KEY=gpu_key,
                   GENERATION_ENGINE="remote_flux", MAKEUP_GENERATION_ENGINE="remote_makeup",
                   NAILS_PREVIEW_MODE="hybrid", NAILS_INFERENCE_STEPS="8",
                   NAILS_HAND_LANDMARKER_PATH=str(landmark), NAILS_SEGMENT_CHECKPOINT=str(checkpoint),
                   NAILS_SEGMENT_PYTHON=str(seg_python), CONSULTATION_PROVIDER="gemini",
                   CONSULTATION_GEMINI_MODEL="gemini-3.5-flash-lite", GEMINI_API_KEY=gemini_key,
                   HAIRCAPSTONE_STYLE_REGISTRY_PATH=str(CPU_ROOT / "backend/app/style_registry_train002_smoke.json"),
                   FRONTEND_ORIGINS="")
    with (WORK / "api.log").open("w") as log:
        api = subprocess.Popen([str(api_python), "-m", "uvicorn", "app.demo_server:app",
                                "--host", "127.0.0.1", "--port", str(PORT), "--workers", "1"],
                               cwd=CPU_ROOT, env=cpu_env, stdout=log, stderr=subprocess.STDOUT,
                               start_new_session=True)
    deadline = time.monotonic() + 120
    while True:
        if api.poll() is not None:
            raise RuntimeError("API exited; inspect api.log")
        try:
            if read_json(f"http://127.0.0.1:{PORT}/health")["status"] == "ok":
                break
        except Exception:
            pass
        if time.monotonic() >= deadline:
            raise RuntimeError("API did not start; inspect api.log")
        time.sleep(2)
    readiness = read_json(f"http://127.0.0.1:{PORT}/deployment/readiness", backend_key)
    if not readiness.get("ready"):
        raise RuntimeError("AI readiness failed; inspect API and GPU logs")
    # Bootstrap already downloaded cloudflared; point only at the authenticated facade.
    binary = WORK / "gpu-session/cloudflared"
    if not binary.exists():
        import shutil
        with urlopen("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", timeout=90) as source, binary.open("wb") as target:
            shutil.copyfileobj(source, target)
        binary.chmod(0o700)
    with (WORK / "tunnel.log").open("w") as log:
        tunnel = subprocess.Popen([str(binary), "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                                  cwd=WORK, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    deadline = time.monotonic() + 240
    while True:
        if tunnel.poll() is not None:
            raise RuntimeError("API tunnel exited; inspect tunnel.log")
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", (WORK / "tunnel.log").read_text())
        if match:
            try:
                if read_json(match[0] + "/deployment/readiness", backend_key).get("ready"):
                    break
            except Exception:
                pass
        if time.monotonic() >= deadline:
            raise RuntimeError("Public API readiness failed; inspect tunnel.log")
        time.sleep(3)
    record = {"status": "READY_FOR_REHEARSAL", "url": match[0], "api_pid": api.pid,
              "tunnel_pid": tunnel.pid, "readiness": readiness, "host_ram_bytes": memory,
              "full_generation_verified": False}
    (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
    website = os.environ.get("BEAUTYCORE_PUBLIC_URL", "").strip()
    if website:
        import importlib.util
        registrar_path = WORK / "runtime_registration.py"
        if not registrar_path.is_file():
            raise RuntimeError("Automatic registration helper is missing; use the corrected client notebook")
        spec = importlib.util.spec_from_file_location("demo_registrar", registrar_path)
        registrar = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(registrar)
        record.update(session_id=session_id, started_at=started_at, status="AI_READY_CONNECTION_PENDING")
        (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
        try:
            ack = registrar.register(website, record, backend_key)
        except registrar.RegistrationError:
            print("AI is running, but website registration was not acknowledged. Inspect registration status; do not rerun GPU startup.", flush=True)
            raise
        record.update(status="CONNECTED_FOR_REHEARSAL", registration_expires_at=ack["expires_at"])
        (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
        with (WORK / "registration.log").open("w") as log:
            heartbeat = subprocess.Popen([sys.executable, str(registrar_path), website, str(WORK / "demo-session.json")],
                env={**env, "AI_BACKEND_API_KEY": backend_key}, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        record["heartbeat_pid"] = heartbeat.pid
        (WORK / "demo-session.json").write_text(json.dumps(record, indent=2))
        print("CONNECTED TO WEBSITE:", website, flush=True)
        print("No Render URL edit is needed. Keep the Kaggle session running.", flush=True)
    else:
        print("Render AI_FASTAPI_URL:", match[0], flush=True)
    print("READY_FOR_REHEARSAL: run the full application checklist before the defense.", flush=True)



main()


In [ ]:
record = json.loads((CLIENT_WORK / 'demo-session.json').read_text())
print('Status:', record['status'])
print('Website:', BEAUTYCORE_PUBLIC_URL)
print('Connection:', record['status'])
print('Readiness is not proof of completed generation; rehearse all features.')
